<a href="https://colab.research.google.com/github/bipasnadulal/aws-future-ai-programmer-nanodegree/blob/main/implement_backpropagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

data-prep

In [2]:
import numpy as np
import pandas as pd

admissions = pd.read_csv('/binary.csv')

# Make dummy variables for rank
data = pd.concat([admissions, pd.get_dummies(admissions['rank'], prefix='rank')], axis=1)
data = data.drop('rank', axis=1)

# Standarize features
for field in ['gre', 'gpa']:
    mean, std = data[field].mean(), data[field].std()
    data[field] = data[field].astype(float)
    data.loc[:, field] = (data[field] - mean) / std

# Split off random 10% of the data for testing
np.random.seed(21)
sample = np.random.choice(data.index, size=int(len(data) * 0.9), replace=False)
data, test_data = data.iloc[sample], data.drop(sample)

# Split into features and targets
features, targets = data.drop('admit', axis=1).astype(float), data['admit']
features_test, targets_test = test_data.drop('admit', axis=1).astype(float), test_data['admit']

In [4]:
import numpy as np


def sigmoid(x):
    """Calculate sigmoid"""
    return 1 / (1 + np.exp(-np.array(x, dtype=float)))


# ─────────────────────────────────────────────
#  SIGMOID NOTEBOOK GRADERS
# ─────────────────────────────────────────────

def grade_h(h, x, w):
    """Grade: linear combination of inputs and weights (h = x · w)"""
    answer = np.dot(x, w)
    if h == answer:
        print(" Good job! `h` is correct.")
    else:
        print(" Try again. `h` is not correct.")


def grade_nn_output(nn_output, x, w):
    """Grade: sigmoid output of neural network"""
    answer = sigmoid(np.dot(x, w))
    if nn_output == answer:
        print(" Good job! `nn_output` is correct.")
    else:
        print(" Try again. `nn_output` is not correct.")


def grade_error(error, y, nn_output, x, w):
    """Grade: error = y - nn_output"""
    nn_output_answer = sigmoid(np.dot(x, w))
    answer = y - nn_output_answer
    if error == answer:
        print(" Good job! `error` is correct.")
    else:
        print(" Try again. `error` is not correct.")


def grade_error_term(error_term, y, x, w):
    """Grade: error_term = error * nn_output * (1 - nn_output)"""
    nn_output_answer = sigmoid(np.dot(x, w))
    error_answer = y - nn_output_answer
    answer = error_answer * nn_output_answer * (1 - nn_output_answer)
    if error_term == answer:
        print(" Good job! `error_term` is correct.")
    else:
        print(" Try again. `error_term` is not correct.")


def grade_del_w(del_w, learnrate, y, x, w):
    """Grade: weight update = learnrate * error_term * x"""
    nn_output_answer = sigmoid(np.dot(x, w))
    error_answer = y - nn_output_answer
    answer = learnrate * error_answer * nn_output_answer * (1 - nn_output_answer) * x
    if np.array_equal(del_w, answer):
        print(" Good job! `del_w` is correct.")
    else:
        print(" Try again. `del_w` is not correct.")


# ─────────────────────────────────────────────
#  BACKPROPAGATION NOTEBOOK GRADERS
# ─────────────────────────────────────────────

def grade_output_error(error, target, output):
    """Grade: output error = target - output"""
    answer = target - output
    if error == answer:
        print(" Well done! `error` is correct.")
    else:
        print(" Try again. `error` is not correct.")


def grade_output_error_term(output_error_term, error, output):
    """Grade: output error term = error * output * (1 - output)"""
    answer = error * output * (1 - output)
    if output_error_term == answer:
        print(" Well done! `output_error_term` is correct.")
    else:
        print(" Try again. `output_error_term` is not correct.")


def grade_hidden_error_term(hidden_error_term, output_error_term, weights_hidden_output, hidden_layer_output):
    """Grade: hidden error term = (output_error_term · W_ho) * h * (1 - h)"""
    answer = (np.dot(output_error_term, weights_hidden_output)
              * hidden_layer_output * (1 - hidden_layer_output))
    if np.array_equal(hidden_error_term, answer):
        print(" Well done! `hidden_error_term` is correct.")
    else:
        print(" Try again. `hidden_error_term` is not correct.")


def grade_delta_w_h_o(delta_w_h_o, learnrate, output_error_term, hidden_layer_output):
    """Grade: Δw (hidden→output) = learnrate * output_error_term * hidden_layer_output"""
    answer = learnrate * output_error_term * hidden_layer_output
    if np.array_equal(delta_w_h_o, answer):
        print(" Well done! `delta_w_h_o` is correct.")
    else:
        print(" Try again. `delta_w_h_o` is not correct.")


def grade_delta_w_i_h(delta_w_i_h, learnrate, hidden_error_term, x):
    """Grade: Δw (input→hidden) = learnrate * hidden_error_term * x (outer product)"""
    answer = learnrate * hidden_error_term * x[:, None]
    if np.array_equal(delta_w_i_h, answer):
        print(" Well done! `delta_w_i_h` is correct.")
    else:
        print(" Try again. `delta_w_i_h` is not correct.")


# ─────────────────────────────────────────────
#  IMPLEMENT BACKPROPAGATION NOTEBOOK GRADER
# ─────────────────────────────────────────────

def grade_implement_backprop(weights_input_hidden, weights_hidden_output,
                              features_test, targets_test, threshold=0.70):
    """
    Grade the full backpropagation implementation by evaluating
    test-set accuracy of the trained weights.

    Parameters
    ----------
    weights_input_hidden  : trained input→hidden weight matrix
    weights_hidden_output : trained hidden→output weight vector
    features_test         : test feature DataFrame / array
    targets_test          : test target Series / array
    threshold             : minimum accuracy to pass (default 0.70)
    """
    hidden = sigmoid(np.dot(features_test, weights_input_hidden))
    out = sigmoid(np.dot(hidden, weights_hidden_output))
    predictions = out > 0.5
    accuracy = np.mean(predictions == targets_test)
    print(f"Prediction accuracy: {accuracy:.3f}")
    if accuracy > threshold:
        print(" Nice job! Your backpropagation implementation is correct.")
    else:
        print(f" Accuracy too low ({accuracy:.3f} < {threshold}). "
              "Check your forward pass, error terms, and weight update steps.")


#### Implementing Backpropagation

In [5]:
import numpy as np

np.random.seed(21)

# Hyperparameters
n_hidden = 2  # number of hidden units
epochs = 900
learnrate = 0.005

n_records, n_features = features.shape
last_loss = None

# Initialize weights
weights_input_hidden = np.random.normal(scale=1 / n_features ** 0.5,
                                        size=(n_features, n_hidden))
weights_hidden_output = np.random.normal(scale=1 / n_features ** 0.5,
                                         size=n_hidden)

In [6]:
for e in range(epochs):
    del_w_input_hidden  = np.zeros(weights_input_hidden.shape)
    del_w_hidden_output = np.zeros(weights_hidden_output.shape)

    for x, y in zip(features.values, targets):

        ## Forward pass ##
        # TODO: Calculate the hidden layer input (linear combination)
        hidden_input = np.dot(x, weights_input_hidden)

        # TODO: Calculate the hidden layer output (apply sigmoid)
        hidden_output = sigmoid(hidden_input)

        # TODO: Calculate the final output (apply sigmoid after linear combination)
        output = sigmoid(np.dot(hidden_output, weights_hidden_output))

        ## Backward pass ##
        # TODO: Calculate the network's prediction error
        error = y - output

        # TODO: Calculate error term for the output unit
        #       Hint: output * (1 - output) is the sigmoid derivative
        output_error_term = error * output * (1-output)

        # TODO: Calculate the hidden layer's contribution to the error
        #       Hint: propagate output_error_term back through weights_hidden_output
        hidden_error = np.dot(output_error_term, weights_hidden_output)

        # TODO: Calculate the error term for the hidden layer
        #       Hint: multiply hidden_error by the sigmoid derivative of hidden_output
        hidden_error_term = hidden_error * hidden_output * (1-hidden_output)

        # TODO: Accumulate the weight updates
        del_w_hidden_output += output_error_term * hidden_output  # replace 0 with your expression
        del_w_input_hidden  += hidden_error_term * np.array(x[:, None], dtype=np.float64)  # replace 0 with your expression

    # TODO: Update weights (remember to divide by n_records)
    weights_input_hidden += learnrate * del_w_input_hidden / n_records
    weights_hidden_output += learnrate * del_w_hidden_output / n_records

    # Print training loss every 10% of epochs
    if e % (epochs / 10) == 0:
        hidden_output = sigmoid(np.dot(x, weights_input_hidden))
        out  = sigmoid(np.dot(hidden_output, weights_hidden_output))
        loss = np.mean((out - targets) ** 2)
        if last_loss and last_loss < loss:
            print("Train loss: ", loss, "  WARNING - Loss Increasing")
        else:
            print("Train loss: ", loss)
        last_loss = loss

# ── Grading ──────────────────────────────────────────────────────────────────
grade_implement_backprop(weights_input_hidden, weights_hidden_output,
                         features_test, targets_test)

Train loss:  0.2513572524259881
Train loss:  0.24996540718842905
Train loss:  0.24862005218904512
Train loss:  0.2473199321717981
Train loss:  0.24606380465584854
Train loss:  0.24485044179257037
Train loss:  0.243678632018683
Train loss:  0.24254718151769472
Train loss:  0.24145491550165454
Train loss:  0.24040067932493334
Prediction accuracy: 0.725
 Nice job! Your backpropagation implementation is correct.
